# Q14: PagedAttention for LLM Throughput
**Topic:** LLM | **Difficulty:** Medium | **Time:** 15 min
**Sheet:** GrindLLM50

---

## Question
What is the primary challenge in LLM throughput optimization? Can you explain how Paged Attention can help?

## Detailed Answer

### The Primary Challenge: KV Cache Memory
During LLM inference, the **KV cache** is the main memory bottleneck:
- Each request needs to store K and V for all layers, all heads, all tokens
- Memory per request scales linearly with sequence length
- With batch serving, KV cache dominates GPU memory
- Different sequences have different lengths → **memory fragmentation**

### The Fragmentation Problem
Standard approach: Pre-allocate contiguous memory for max_seq_len per request.
- Request needing 100 tokens → allocate 2048 tokens of KV space (wasted!)
- **50-70% of KV cache memory is wasted** on average (internal + external fragmentation)

### PagedAttention (vLLM)
Inspired by OS virtual memory and paging.

**Idea**: Store KV cache in non-contiguous **pages** (blocks) and use a page table.

```
Traditional: [Request 1 KV (contiguous)] [Wasted] [Request 2 KV (contiguous)] [Wasted]
Paged:       [Page1-R1][Page1-R2][Page2-R1][Page2-R2][Page3-R1]...
```

Each page holds a fixed number of KV pairs (block_size=16 tokens typically).

### How It Works
1. Divide KV cache into fixed-size **blocks** (pages)
2. Maintain a **page table** (block table) mapping logical position → physical block
3. Allocate blocks **on demand** as sequence grows
4. No pre-allocation waste — only allocate what's needed
5. Freed blocks are instantly reusable

### Benefits
| Benefit | Impact |
|---------|--------|
| Near-zero waste | ~0% vs 50-70% internal fragmentation |
| Higher batch sizes | 2-4x more requests per GPU |
| Copy-on-write | Efficient beam search (share KV pages) |
| Dynamic allocation | No need to predict max length |

## Key Takeaways
- KV cache **memory fragmentation** is the primary throughput bottleneck
- PagedAttention applies OS paging concepts to KV cache — near-zero waste
- Enables **2-4x higher throughput** by fitting more requests in memory
- **vLLM** implements PagedAttention and is the standard for LLM serving